## 추론

In [1]:
import pandas as pd
import os
import re
import json
import yaml
from datetime import datetime
from glob import glob
from tqdm import tqdm
from pprint import pprint
import torch
import pytorch_lightning as pl
from rouge import Rouge # 모델의 성능을 평가하기 위한 라이브러리입니다.

from torch.utils.data import Dataset , DataLoader
from transformers import AutoTokenizer, BartForConditionalGeneration, BartConfig
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
from transformers import AutoConfig, AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq


import wandb # 모델 학습 과정을 손쉽게 Tracking하고, 시각화할 수 있는 라이브러리입니다.

In [2]:
MODEL_NAME = "paust/pko-t5-base"
MODEL_NAME_SHORT = "paust"
VERSION = "V1"

# 저장된 config 파일을 불러옵니다.
config_path = f"./config_{MODEL_NAME_SHORT}_{VERSION}.yaml"
with open(config_path, "r") as file:
    loaded_config = yaml.safe_load(file)

In [3]:
# 가장 최신의 run name 가져옴
run_name = MODEL_NAME_SHORT+"/"+sorted(os.listdir("./checkpoint"))[-1]
print(f"✅ 자동 선택된 최신 run name: {run_name}")

✅ 자동 선택된 최신 run name: paust/pko-t5-base_V2_1208_0223


### 1) rouge 점수가 가장 높은 체크포인트 찾기

In [3]:
# CHECKPOINT_DIR = os.path.join(loaded_config['inference']['ckt_path'], run_name.split('/')[1])
CHECKPOINT_DIR = os.path.join(loaded_config['inference']['ckt_path'])
ROUGE_KEYS = ["eval_rouge-1", "eval_rouge-2", "eval_rouge-l"]

def auto_select_best_checkpoint(CHECKPOINT_DIR=CHECKPOINT_DIR):
    best_score = -1
    best_path = None

    for ckpt_name in os.listdir(CHECKPOINT_DIR):
        ckpt_path = os.path.join(CHECKPOINT_DIR, ckpt_name)
        trainer_state_path = os.path.join(ckpt_path, "trainer_state.json")

        if not os.path.isfile(trainer_state_path):
            continue

        # trainer_state.json 읽기
        with open(trainer_state_path, "r") as f:
            state = json.load(f)

        # metrics = 마지막 저장 시점의 평가 점수
        metrics = state.get("log_history", [])
        
        # log_history 중 evaluation 항목만 추출
        for m in metrics:
            if "eval_loss" in m:  # eval 스텝의 로깅
                # 필요한 ROUGE 값 전부 있는지 확인
                if all(k in m for k in ROUGE_KEYS):
                    rouge1 = m["eval_rouge-1"]
                    rouge2 = m["eval_rouge-2"]
                    rougel = m["eval_rouge-l"]

                    # 종합 점수 계산 (단순 평균)
                    final_score = (rouge1 + rouge2 + rougel) / 3

                    # 최고의 체크포인트 갱신
                    if final_score > best_score:
                        best_score = final_score
                        best_path = ckpt_path

    print("Best checkpoint:", best_path)
    print("Best ROUGE score:", best_score)

    return best_path


In [4]:
loaded_config['inference']['ckt_path'] = auto_select_best_checkpoint()

Best checkpoint: ./checkpoint/checkpoint-1945
Best ROUGE score: 0.19217661547394316


### 2) test data를 사용하여 모델의 성능을 확인

In [5]:
# ------------------------
# Dataset (ID 제거!)
# ------------------------
class DatasetForInference(Dataset):
    def __init__(self, encoder_input, data_len):
        self.encoder_input = encoder_input
        self.data_len = data_len

    def __len__(self):
        return self.data_len

    def __getitem__(self, idx):
        # ID 넣지 않는다 — collator가 텐서로 변환하려고 해서 오류 발생하니까!
        item = {key: val[idx] for key, val in self.encoder_input.items()}
        return item

In [6]:
# tokenization 과정까지 진행된 최종적으로 모델에 입력될 데이터를 출력합니다.
def prepare_test_dataset(config,preprocessor, tokenizer):

    test_file_path = os.path.join(config['general']['data_path'],'test_preprocessed.csv')
    print(test_file_path)
    test_data = preprocessor.make_set_as_df(test_file_path,is_train=False)
    test_id = test_data['fname']

    print('-'*150)
    print(f'test_data:\n{test_data["dialogue"][0]}')
    print('-'*150)

    encoder_input_test, _ = preprocessor.make_input(test_data, is_test=True)
    print('-'*10, 'Load data complete', '-'*10,)

    test_tokenized_encoder_inputs = tokenizer(
        encoder_input_test, 
        # return_tensors="pt", 
        return_tensors=None, 
        padding=False,
        add_special_tokens=True, 
        truncation=True, 
        max_length=config['tokenizer']['encoder_max_len'], 
        return_token_type_ids=False,)

    test_encoder_inputs_dataset = DatasetForInference(test_tokenized_encoder_inputs, test_id, len(encoder_input_test))
    print('-'*10, 'Make dataset complete', '-'*10,)

    return test_data, test_encoder_inputs_dataset

In [7]:
# 추론을 위한 tokenizer와 학습시킨 모델을 불러옵니다.
def load_tokenizer_and_model_for_test(config,device):
    print('-'*10, 'Load tokenizer & model', '-'*10,)

    model_name = config['general']['model_name']
    ckt_path = config['inference']['ckt_path']
    print('-'*10, f'Model Name : {model_name}', '-'*10,)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    special_tokens_dict = {'additional_special_tokens': config['tokenizer']['special_tokens']}
    tokenizer.add_special_tokens(special_tokens_dict)

    generate_model = AutoModelForSeq2SeqLM.from_pretrained(ckt_path)
    generate_model.resize_token_embeddings(len(tokenizer))
    generate_model.to(device)
    
    print('-'*10, 'Load tokenizer & model complete', '-'*10,)

    return generate_model , tokenizer

In [8]:
# 데이터 전처리를 위한 클래스로, 데이터셋을 데이터프레임으로 변환하고 인코더와 디코더의 입력을 생성합니다.
class Preprocess:
    def __init__(self,
            # bos_token: str, TODO
            eos_token: str,
        ) -> None:

        # self.bos_token = bos_token TODO
        self.eos_token = eos_token

    @staticmethod
    # 실험에 필요한 컬럼을 가져옵니다.
    def make_set_as_df(file_path, is_train = True):
        if is_train:
            df = pd.read_csv(file_path)
            train_df = df[['fname','dialogue','summary']]
            return train_df
        else:
            df = pd.read_csv(file_path)
            test_df = df[['fname','dialogue']]
            return test_df

    # 입력, 출력 형태를 맞추기 위해 전처리를 진행합니다.
    def make_input(self, dataset, is_test = False):
        encoder_input = dataset['dialogue'].apply(lambda x: str(x))
        if is_test:
            decoder_input = [""] * len(dataset)
            # decoder_input = [self.bos_token] * len(dataset['dialogue']) TODO
            return encoder_input.tolist(), list(decoder_input)
        else:
            # TODO
            # decoder_input = dataset['summary'].apply(lambda x : self.bos_token + str(x)) # Ground truth를 디코더의 input으로 사용하여 학습합니다.
            # decoder_output = dataset['summary'].apply(lambda x : str(x) + self.eos_token)
            # return encoder_input.tolist(), decoder_input.tolist(), decoder_output.tolist()
            decoder_input = dataset['summary'].apply(lambda x: str(x))
            decoder_output = dataset['summary'].apply(lambda x: str(x) + self.eos_token)
            return (
                encoder_input.tolist(),
                decoder_input.tolist(),
                decoder_output.tolist()
            )

### 3) 간단한 후처리

In [9]:
def clean_summary(text):

    text = text.strip()

    # 1. 앞뒤 쌍따옴표 제거
    if len(text) >= 2 and text[0] == '"' and text[-1] == '"':
        text = text[1:-1].strip()

    # 2. 전체 문자열 내 쌍따옴표 제거
    text = text.replace('"', '')

    # 3. 중복 공백 제거
    text = re.sub(r"\s+", " ", text)

    # 4. 문장부호 앞 공백 제거
    text = re.sub(r"\s+([,.!?])", r"\1", text)

    # 5. #Person 토큰 앞 공백 제거
    text = re.sub(r"#Person(\d+)#\s+", r"#Person\1#", text)

    return text

### 4) 추론 수행

In [10]:
# ------------------------
# Inference 함수
# ------------------------
def inference(config):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    # 모델 & 토크나이저 불러오기
    generate_model, tokenizer = load_tokenizer_and_model_for_test(config,device)

    test_file_path = os.path.join(config['general']['data_path'],'test_preprocessed.csv')
    test_data = pd.read_csv(test_file_path)
    test_id_list = test_data["fname"].tolist()

    # Preprocess 적용
    preprocessor = Preprocess(config['tokenizer']['eos_token'])
    encoder_input_test, _ = preprocessor.make_input(test_data, is_test=True)

    # 문자열 강제 변환 (nested list 방지)
    encoder_input_test = [str(x) for x in encoder_input_test]

    # Tokenization — return_tensors 절대 넣지 않는다!
    encodings = tokenizer(
        encoder_input_test,
        padding=False,
        truncation=True,
        max_length=config["tokenizer"]["encoder_max_len"],
        add_special_tokens=True,
        return_token_type_ids=False
    )

    # Dataset 생성 (ID 없음)
    test_dataset = DatasetForInference(encodings, len(test_id_list))

    # Collator
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=generate_model,
        pad_to_multiple_of=8
    )

    # DataLoader
    dataloader = DataLoader(
        test_dataset,
        batch_size=config["inference"]["batch_size"],
        shuffle=False,
        collate_fn=collator
    )

    print("Inference 시작!")

    summaries = []
    aligned_ids = []

    # ---------------------
    # Inference Loop
    # ---------------------
    with torch.no_grad():
        global_index = 0  # test_id_list와 매칭하기 위한 인덱스

        for batch in tqdm(dataloader):
            # batch['input_ids'], batch['attention_mask'] — collator가 텐서로 만들어준다
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            batch_size = input_ids.shape[0]

            # ID를 batch 크기만큼 다시 매칭
            aligned_ids.extend(test_id_list[global_index : global_index + batch_size])
            global_index += batch_size

            # Generate
            generated = generate_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                no_repeat_ngram_size=config["inference"]["no_repeat_ngram_size"],
                early_stopping=config["inference"]["early_stopping"],
                max_length=config["inference"]["generate_max_length"],
                num_beams=config["inference"]["num_beams"]
            )

            # Decode
            for ids in generated:
                text = tokenizer.decode(ids, skip_special_tokens=False).strip()
                text = text.replace("<pad>", "").strip()
                summaries.append(text)
    
    # 정확한 평가를 위하여 노이즈에 해당되는 스페셜 토큰을 제거합니다.
    remove_tokens = config['inference']['remove_tokens']
    preprocessed_summary = summaries.copy()
    for token in remove_tokens:
        preprocessed_summary = [sentence.replace(token," ") for sentence in preprocessed_summary]

    # 추가적인 후처리
    preprocessed_summary = [clean_summary(text) for text in preprocessed_summary]

    output = pd.DataFrame(
        {
            "fname": test_data['fname'],
            "summary" : preprocessed_summary,
        }
    )

    # 결과 반환
    return output

In [11]:
# 학습된 모델의 test를 진행합니다.
if __name__ == "__main__":
    output = inference(loaded_config)

---------- Load tokenizer & model ----------
---------- Model Name : paust/pko-t5-base ----------
---------- Load tokenizer & model complete ----------
Inference 시작!


100%|██████████| 32/32 [01:16<00:00,  2.38s/it]


In [12]:
result_path = loaded_config['inference']['result_path']
if not os.path.exists(result_path):
    os.makedirs(result_path)

from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"output_{timestamp}.csv"

save_path = os.path.join(result_path, filename)
output.to_csv(save_path, index=False)

print(f"✅ 저장 경로: {save_path}")

✅ 저장 경로: ./prediction/output_20251208_053303.csv
